# 01 · Translate English SMS → Sinhala & Tamil (NLLB-200)

**Why:** the Step 6 baselines raised 26-31% false alarms on *normal* Sinhala/Tamil messages they had never seen. This notebook adds ~2,400 translated training messages - mostly **normal** ones - to fix that.

**How to run (≈15 min):**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. When asked, upload **`D:\ladhu\smishing-detector\data\processed\train.csv`**
4. At the end two files download automatically → put both in **`D:\ladhu\smishing-detector\data\raw\`**
   - `multilingual_mt.csv` (the translations)
   - `mt_review_sample.csv` (60 rows for a native-speaker spot check)

Only English rows of the **training** split are translated, and `data_prep.py` adds them to TRAIN only, so nothing leaks into the test set.


In [ ]:
!pip install -q transformers sentencepiece
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4 GPU!')

In [ ]:
from google.colab import files
up = files.upload()                      # choose data/processed/train.csv
import io, pandas as pd
train = pd.read_csv(io.BytesIO(next(iter(up.values()))))
print(train.shape); print(train.groupby(['language','label3']).size().unstack(fill_value=0))

In [ ]:
SEED = 42
N_HAM, N_SPAM = 900, 150            # normal messages are what Sinhala/Tamil lacks most
TARGETS = {'si': 'sin_Sinh', 'ta': 'tam_Taml'}
MODEL = 'facebook/nllb-200-distilled-600M'

en = train[train.language == 'en']
src = pd.concat([
    en[en.label3 == 'smishing'],                                               # all English smishing
    en[en.label3 == 'spam'].sample(min(N_SPAM, (en.label3 == 'spam').sum()), random_state=SEED),
    en[en.label3 == 'ham'][en.text.str.len().between(15, 300)].sample(N_HAM, random_state=SEED),
]).reset_index(drop=True)
print(src.label3.value_counts())

In [ ]:
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# protect links / e-mails / long numbers:  'visit bit.ly/x' -> 'visit ZQX0' -> translate -> put back
PROTECT = re.compile(r'(https?://\S+|www\.\S+|\b[\w-]+\.(?:com|lk|net|org|info|co|uk|xyz|top|buzz|site|online|help|icu|cc|me|ly)(?:/\S*)?|\S+@\S+|\+?\d[\d\s-]{6,}\d)', re.I)
def mask(t):
    found = PROTECT.findall(t)
    for i, f in enumerate(found): t = t.replace(f, f'ZQX{i}', 1)
    return t, found
def unmask(t, found):
    for i, f in enumerate(found):
        t = t.replace(f'ZQX{i}', f, 1) if f'ZQX{i}' in t else t + ' ' + f
    return re.sub(r'ZQX\d+', '', t).strip()

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL, src_lang='eng_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=torch.float16 if dev == 'cuda' else torch.float32).to(dev).eval()

@torch.no_grad()
def translate(texts, tgt, bs=48):
    out = []
    for i in range(0, len(texts), bs):
        batch = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=200).to(dev)
        gen = model.generate(**batch, forced_bos_token_id=tok.convert_tokens_to_ids(tgt), max_new_tokens=256, num_beams=4)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'{tgt}: {min(i+bs, len(texts))}/{len(texts)}', end='\r')
    print()
    return out

In [ ]:
SCRIPT = {'si': re.compile(r'[\u0D80-\u0DFF]'), 'ta': re.compile(r'[\u0B80-\u0BFF]')}
def good(t, lang):
    letters = [c for c in t if c.isalpha()]
    if len(letters) < 5: return False
    ratio = sum(bool(SCRIPT[lang].match(c)) for c in letters) / len(letters)
    words = t.split()
    looping = len(words) > 6 and len(set(words)) / len(words) < 0.4
    return ratio >= 0.5 and not looping

masked = [mask(t) for t in src.text]
rows = []
for lang, code_ in TARGETS.items():
    hyp = translate([m for m, _ in masked], code_)
    for (m, found), h, (_, r) in zip(masked, hyp, src.iterrows()):
        t = unmask(h, found)
        if good(t, lang):
            rows.append(dict(text=t, label=r.label3, language=lang, script='native', category='translated',
                             origin='mt_nllb600m', src_text=r.text))
mt = pd.DataFrame(rows).drop_duplicates('text')
print('kept', len(mt), 'of', 2 * len(src)); print(pd.crosstab(mt.language, mt.label))

In [ ]:
review = mt.groupby('language', group_keys=False).apply(lambda g: g.sample(30, random_state=SEED))
review = review[['language', 'label', 'src_text', 'text']].assign(ok_yes_no='', note='')
mt.to_csv('multilingual_mt.csv', index=False, encoding='utf-8')
review.to_csv('mt_review_sample.csv', index=False, encoding='utf-8-sig')   # utf-8-sig opens correctly in Excel
display(review.head(8))
files.download('multilingual_mt.csv'); files.download('mt_review_sample.csv')